## Notebook 04 — Delta MERGE (Incremental CDC Load)

**Pattern:** Simulate a monthly incremental load using Delta MERGE
- Some providers have updated billing numbers (MATCHED → UPDATE)
- Some providers are brand new (NOT MATCHED → INSERT)
- Verify with DESCRIBE HISTORY and Delta time travel

## Configuration


In [0]:
CATALOG = "adb_retailedge_dev"
SCHEMA  = "healthcare_cms"
TARGET  = f"{CATALOG}.{SCHEMA}.silver_provider_summary"

print(f"Target table : {TARGET}")

Target table : adb_retailedge_dev.healthcare_cms.silver_provider_summary


## Snapshot row count BEFORE merge

In [0]:
df_before = spark.table(TARGET)
count_before = df_before.count()
print(f"Rows Before merge: {count_before:,}")

Rows Before merge: 73,678


## simulate incoming "new month" data
- This cell creates the incoming batch — 200 existing providers with updated payment values + 5 brand new providers

In [0]:
  from pyspark.sql.functions import col, round as spark_round
  from pyspark.sql import Row

  fake_npis = ["9990000001","9990000002","9990000003","9990000004","9990000005"]

  # Exclude fake NPIs so no overlap with new_providers
  df_updates = spark.table(TARGET) \
      .filter(~col("provider_npi").isin(fake_npis)) \
      .limit(200) \
      .withColumn("avg_medicare_payment", spark_round(col("avg_medicare_payment") * 1.05, 2)) \
      .withColumn("avg_submitted_charge", spark_round(col("avg_submitted_charge") * 1.03, 2)) \
      .withColumn("avg_allowed_amount",   spark_round(col("avg_allowed_amount")   * 1.04, 2)) \
      .withColumn("total_services",       spark_round(col("total_services")       * 1.10, 2)) \
      .withColumn("total_beneficiaries",  (col("total_beneficiaries") * 1.08).cast("long")) \
      .withColumn("total_procedures",     (col("total_procedures") + 5).cast("long"))

  # 5 brand new providers
  new_providers = spark.createDataFrame([
      Row(provider_npi="9990000001", provider_name="NEW HEALTH CLINIC",     provider_first_name="JAMES",
          city="AUSTIN",   state="TX", zip_code="78701", provider_type="Internal Medicine",
          total_procedures=12, total_beneficiaries=320,  total_services=980.0,
          avg_medicare_payment=185.50, avg_submitted_charge=310.00, avg_allowed_amount=210.75),
      Row(provider_npi="9990000002", provider_name="SUNRISE MEDICAL GROUP", provider_first_name="SARA",
          city="PHOENIX",  state="AZ", zip_code="85001", provider_type="Family Practice",
          total_procedures=8,  total_beneficiaries=210,  total_services=640.0,
          avg_medicare_payment=162.00, avg_submitted_charge=275.00, avg_allowed_amount=190.20),
      Row(provider_npi="9990000003", provider_name="VALLEY CARE CENTER",    provider_first_name="MICHAEL",
          city="DENVER",   state="CO", zip_code="80201", provider_type="Cardiology",
          total_procedures=20, total_beneficiaries=450,  total_services=1200.0,
          avg_medicare_payment=320.00, avg_submitted_charge=510.00, avg_allowed_amount=365.00),
      Row(provider_npi="9990000004", provider_name="LAKESIDE PHYSICIANS",   provider_first_name="EMILY",
          city="CHICAGO",  state="IL", zip_code="60601", provider_type="Orthopedic Surgery",
          total_procedures=15, total_beneficiaries=280,  total_services=890.0,
          avg_medicare_payment=425.00, avg_submitted_charge=720.00, avg_allowed_amount=490.00),
      Row(provider_npi="9990000005", provider_name="COAST MEDICAL ASSOC",   provider_first_name="DAVID",
          city="SEATTLE",  state="WA", zip_code="98101", provider_type="Neurology",
          total_procedures=18, total_beneficiaries=390,  total_services=1100.0,
          avg_medicare_payment=380.00, avg_submitted_charge=620.00, avg_allowed_amount=430.00),
  ])

  df_incoming = df_updates.union(new_providers)

  print(f"Incoming batch size : {df_incoming.count():,} rows")
  print(f"  - Updated records : 200")
  print(f"  - New records     : 5")

Incoming batch size : 205 rows
  - Updated records : 200
  - New records     : 5


## Register incoming data as a Temp View

- **Why we do this:** Delta MERGE uses SQL syntax. SQL can only read from tables or views — it cannot directly read a PySpark DataFrame. So we register df_incoming as a temporary view so the MERGE SQL in the next cell can access it

In [0]:
df_incomming.createOrReplaceTempView("incoming_providers")
print("Temp view 'incoming_providers' registered")

Temp view 'incoming_providers' registered


In [0]:
   # Write source to staging table
  df_incoming.write \
      .format("delta") \
      .mode("overwrite") \
      .option("overwriteSchema", "true") \
      .saveAsTable(f"{CATALOG}.{SCHEMA}.staging_incoming_providers")

  count = spark.table(f"{CATALOG}.{SCHEMA}.staging_incoming_providers").count()
  print(f"Staging table written: {count:,} rows")

Staging table written: 205 rows


## Execute Delta MERGE

In [0]:
  from pyspark.sql.functions import count

  # Find any duplicate NPIs in the incoming batch
  duplicates = df_incoming \
      .groupBy("provider_npi") \
      .agg(count("*").alias("cnt")) \
      .filter("cnt > 1")

  dup_count = duplicates.count()
  print(f"Duplicate NPIs in source: {dup_count}")

  if dup_count > 0:
      display(duplicates)

Duplicate NPIs in source: 0


In [0]:
  spark.sql(f"""
      MERGE INTO {TARGET} AS target
      USING {CATALOG}.{SCHEMA}.staging_incoming_providers AS source
      ON target.provider_npi = source.provider_npi

      WHEN MATCHED THEN UPDATE SET
          target.avg_medicare_payment = source.avg_medicare_payment,
          target.avg_submitted_charge = source.avg_submitted_charge,
          target.avg_allowed_amount   = source.avg_allowed_amount,
          target.total_services       = source.total_services,
          target.total_beneficiaries  = source.total_beneficiaries,
          target.total_procedures     = source.total_procedures

      WHEN NOT MATCHED THEN INSERT (
          provider_npi,
          provider_name,
          provider_first_name,
          city,
          state,
          zip_code,
          provider_type,
          total_procedures,
          total_beneficiaries,
          total_services,
          avg_medicare_payment,
          avg_submitted_charge,
          avg_allowed_amount
      ) VALUES (
          source.provider_npi,
          source.provider_name,
          source.provider_first_name,
          source.city,
          source.state,
          source.zip_code,
          source.provider_type,
          source.total_procedures,
          source.total_beneficiaries,
          source.total_services,
          source.avg_medicare_payment,
          source.avg_submitted_charge,
          source.avg_allowed_amount
      )
  """)

  print("MERGE complete.")

MERGE complete.


## Verify row counts after merge

In [0]:
  count_after = spark.table(TARGET).count()
  new_rows    = count_after - count_before

  print(f"Rows BEFORE merge : {count_before:,}")
  print(f"Rows AFTER  merge : {count_after:,}")
  print(f"Net new rows      : {new_rows:,}  (expected: 0 — all 5 already exist)")

Rows BEFORE merge : 73,678
Rows AFTER  merge : 73,678
Net new rows      : 0  (expected: 0 — all 5 already exist)


## DESCRIBE HISTORY (Delta transaction log)

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {TARGET}"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-08-12T13:30:26.000Z,146678202426002,3724dd7d-e4eb-49f7-b651-df424f0a3e97,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [], zOrderBy -> [], batchId -> 0)","List(229113798479872, Predictive Optimization Job-5222cadb-31e2-4b3d-b85f-4bd8510a0873, 10901074836925, 1085250388547921, 146678202426002, manual, 7405607062784164)",null,dc1a2327-e017-4a46-8f9a-b64674d46118,0812-132549-2nl3s786-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 21, numRemovedBytes -> 2353168, p25FileSize -> 2243753, numDeletionVectorsRemoved -> 1, minFileSize -> 2243753, numAddedFiles -> 1, maxFileSize -> 2243753, p75FileSize -> 2243753, p50FileSize -> 2243753, numAddedBytes -> 2243753)",null,Databricks-Runtime/18.x-photon-scala2.13
3,2026-08-12T12:57:39.000Z,147351882412252,madhuravenkatesh761@gmail.com,MERGE,"Map(predicate -> [""(provider_npi#11508 = provider_npi#11408)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(212468060050787),e122b141-037a-4e44-8d77-678af38d7845,0812-125453-n4sq80w8-v2n,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 12, numTargetBytesAdded -> 60628, numTargetBytesRemoved -> 18853, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 205, executionTimeMs -> 11199, materializeSourceTimeMs -> 1173, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 1, scanTimeMs -> 4375, numTargetRowsUpdated -> 205, numOutputRows -> 205, numTargetDeletionVectorsRemoved -> 1, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 205, numTargetFilesRemoved -> 5, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 5486)",null,Databricks-Runtime/18.x-photon-scala2.13
2,2026-08-12T10:24:00.000Z,147351882412252,madhuravenkatesh761@gmail.com,MERGE,"Map(predicate -> [""(provider_npi#11510 = provider_npi#11408)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(212468060050787),632e8b90-6a6b-4661-9d4e-5c0981482e0e,0812-093859-bbdvvbfg-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 13, numTargetBytesAdded -> 65175, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 200, executionTimeMs -> 9108, materializeSourceTimeMs -> 1237, numTargetRowsInserted -> 5, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 3434, numTargetRowsUpdated -> 200, numOutputRows -> 205, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 205, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 4310)",null,Databricks-Runtime/18.x-photon-scala2.13
1,2026-08-10T06:28:27.000Z,147351882412252,madhuravenkatesh761@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2653616316995303),2690bc22-2a5c-4aa1-9c92-90ff6a27c9af,0810-060147-77hk3tgz-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 2246218, numDeletionVectorsRemoved -> 0, numOutputRows -> 73673, numOutputBytes -> 2246218)",null,Databricks-Runtime/18.x-photon-scala2.13
0,2026-08-10T06:20:01.000Z,1473518824122

## Find version numbers from history



In [0]:

history_df = spark.sql(f"DESCRIBE HISTORY {TARGET}")
latest_version = history_df.selectExpr("max(version)").collect()[0][0]
pre_merge_version = latest_version - 1

print(f"Latest version    : {latest_version}  (post-merge)")
print(f"Pre-merge version : {pre_merge_version}")

## Time travel : query pre-merge state

In [0]:
df_time_travel = spark.sql(f"""
    SELECT provider_npi, provider_name, state, avg_medicare_payment
    from {TARGET} VERSION AS OF {pre_merge_version} limit 5
    """)

print(f"Pre-merge snapshot (version {pre_merge_version}):")
display(df_time_travel)

## Confirm new providers across versions

In [0]:
  new_npi_list = "('9990000001','9990000002','9990000003','9990000004','9990000005')"

  count_current   = spark.sql(f"""
      SELECT COUNT(*) FROM {TARGET}
      WHERE provider_npi IN {new_npi_list}
  """).collect()[0][0]

  count_pre_merge = spark.sql(f"""
      SELECT COUNT(*) FROM {TARGET} VERSION AS OF 1
      WHERE provider_npi IN {new_npi_list}
  """).collect()[0][0]

  print(f"New providers in current version  : {count_current}   (expected: 5)")
  print(f"New providers in version 1        : {count_pre_merge}  (expected: 0)")

In [0]:
# Spot check: compare one provider before and after MERGE

# What we are checking: Pick one existing provider NPI and compare their payment numbers between version 1 (before MERGE) and current (after MERGE). The numbers should be slightly higher in the current version because we multiplied them by 1.05, 1.03, and 1.10.

sample_npi = spark.sql(f"""
      SELECT provider_npi FROM {TARGET} VERSION AS OF 1
      LIMIT 1
  """).collect()[0][0]

print(f"Spot-checking NPI: {sample_npi}\n")

print("--- BEFORE merge (version 1) ---")
display(spark.sql(f"""
      SELECT provider_npi, avg_medicare_payment, avg_submitted_charge, total_services
      FROM {TARGET} VERSION AS OF 1
      WHERE provider_npi = '{sample_npi}'
  """))

print("--- AFTER merge (current) ---")
display(spark.sql(f"""
      SELECT provider_npi, avg_medicare_payment, avg_submitted_charge, total_services
      FROM {TARGET}
      WHERE provider_npi = '{sample_npi}'
  """))

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7251909174849354>, line 1
----> 1 df_before.explain("formatted")

NameError: name 'df_before' is not defined